# 04C — Validation verrouillée du batch 3 et reconstruction spatiale

Ce notebook exécute les tâches 31 à 33. Il refitte les configurations gelées sur les batches 1–2, projette exclusivement le batch 3, produit les cartes pixels brutes et post-traitées avec le verrou 03C, puis applique des garde-fous préspécifiés sans score composite. Les tracks non soutenus restent des résultats diagnostiques explicites. Le batch 4 n'est jamais chargé.

## A — Initialisation

In [1]:
from __future__ import annotations

import json
import platform
import sys
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

CURRENT_DIR = Path.cwd().resolve()
if (CURRENT_DIR / "src").exists():
    PROJECT_ROOT = CURRENT_DIR
elif (CURRENT_DIR.parent / "src").exists():
    PROJECT_ROOT = CURRENT_DIR.parent
else:
    raise RuntimeError("Launch the notebook from the project root or notebooks/.")
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

pd.set_option("display.max_columns", 24)
pd.set_option("display.max_rows", 30)

from src import experiment_config as expcfg
from src.io.database_h5 import load_nir_uco_h5
from src.protocol_governance import sha256_dataframe, sha256_file
from src.spectra.band_selection import select_wavelength_range_from_database
from src.utils import load_parquet, save_parquet
from src.workflows.simca import (
    run_locked_simca_validation_refit,
    run_locked_simca_validation_refit_checkpointed,
)
from src.workflows.simca_candidates import (
    build_locked_validation_candidate_pool,
    hash_locked_validation_evaluation_rule,
    hash_locked_validation_plan,
)
from src.workflows.simca_grid_evaluation import (
    build_validation_guardrails,
    evaluate_locked_validation_predictions,
)
from src.workflows.spatial_postprocessing_calibration import (
    build_locked_spatial_validation_outputs,
    verify_spatial_postprocessing_lock,
)

%load_ext autoreload
%autoreload 2

print("Python:", platform.python_version())
print("PROJECT_ROOT:", PROJECT_ROOT)

Python: 3.14.6
PROJECT_ROOT: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts


## B — Chemins et plan de validation centralisé

In [2]:
USE_WAVELENGTH_WINDOW = expcfg.USE_WAVELENGTH_WINDOW
RESULTS_TAG = (
    f"{int(expcfg.WAVELENGTH_WINDOW_MIN_NM)}_{int(expcfg.WAVELENGTH_WINDOW_MAX_NM)}"
    if USE_WAVELENGTH_WINDOW else expcfg.DEFAULT_RESULTS_TAG
)
DB_H5_PATH = PROJECT_ROOT.joinpath(*expcfg.DATABASE_H5_RELATIVE_PATH)
CALIBRATION_DIR = PROJECT_ROOT / "results" / f"{expcfg.INTERNAL_CALIBRATION_RESULTS_DIR_PREFIX}_{RESULTS_TAG}"
DOMAIN_DIR = PROJECT_ROOT / "results" / f"{expcfg.DOMAIN_SPATIAL_CALIBRATION_RESULTS_DIR_PREFIX}_{RESULTS_TAG}"
GRID_DIR = PROJECT_ROOT / "results" / f"{expcfg.SIMCA_GRID_SEARCH_RESULTS_DIR_PREFIX}_{RESULTS_TAG}"
OPTUNA_DIR = PROJECT_ROOT / "results" / f"{expcfg.SIMCA_OPTUNA_RESULTS_DIR_PREFIX}_{RESULTS_TAG}"
OUTPUT_DIR = PROJECT_ROOT / "results" / f"{expcfg.SIMCA_CONCAT_REFIT_RESULTS_DIR_PREFIX}_{RESULTS_TAG}"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_PATHS = {
    name: OUTPUT_DIR / filename
    for name, filename in expcfg.SIMCA_CONCAT_REFIT_OUTPUT_FILENAMES.items()
}
CHECKPOINT_DIR = OUTPUT_DIR / expcfg.SIMCA_CONCAT_REFIT_CHECKPOINT_DIRNAME

CALIBRATED_PATH = CALIBRATION_DIR / expcfg.INTERNAL_CALIBRATION_OUTPUT_FILENAMES["calibrated_hyperparameters"]
CALIBRATION_DOMAIN_PATH = CALIBRATION_DIR / expcfg.INTERNAL_CALIBRATION_OUTPUT_FILENAMES["calibration_domain"]
TRACK_CONTRACTS_PATH = CALIBRATION_DIR / expcfg.INTERNAL_CALIBRATION_OUTPUT_FILENAMES["track_contracts"]
PROJECTION_ELIGIBILITY_PATH = DOMAIN_DIR / expcfg.DOMAIN_SPATIAL_CALIBRATION_OUTPUT_FILENAMES["projection_eligibility"]
SPATIAL_METRICS_PATH = DOMAIN_DIR / expcfg.DOMAIN_SPATIAL_CALIBRATION_OUTPUT_FILENAMES["spatial_calibration_metrics"]
FRAGMENT_CLASSES_PATH = DOMAIN_DIR / expcfg.DOMAIN_SPATIAL_CALIBRATION_OUTPUT_FILENAMES["fragment_size_classes"]
SPATIAL_LOCK_PATH = DOMAIN_DIR / expcfg.DOMAIN_SPATIAL_CALIBRATION_OUTPUT_FILENAMES["spatial_postprocessing_lock"]
GRID_PARETO_PATH = GRID_DIR / expcfg.SIMCA_GRID_SEARCH_OUTPUT_FILENAMES["pareto_reference"]
GRID_PROTOCOL_PATH = GRID_DIR / expcfg.SIMCA_GRID_SEARCH_OUTPUT_FILENAMES["protocol"]
OPTUNA_TRIALS_PATH = OPTUNA_DIR / expcfg.SIMCA_OPTUNA_OUTPUT_FILENAMES["trials"]
OPTUNA_CANDIDATES_PATH = OPTUNA_DIR / expcfg.SIMCA_OPTUNA_OUTPUT_FILENAMES["pareto_candidates"]
OPTUNA_ABLATION_PATH = OPTUNA_DIR / expcfg.SIMCA_OPTUNA_OUTPUT_FILENAMES["ablation_plan"]
OPTUNA_PROTOCOL_PATH = OPTUNA_DIR / expcfg.SIMCA_OPTUNA_OUTPUT_FILENAMES["protocol"]

VALIDATION_PLAN_HASH = hash_locked_validation_plan()
VALIDATION_EVALUATION_RULE_HASH = hash_locked_validation_evaluation_rule()
previous_protocol = {}
if OUTPUT_PATHS["protocol"].exists():
    previous_protocol = json.loads(OUTPUT_PATHS["protocol"].read_text(encoding="utf-8"))
superseded_output_sha256 = {}
if previous_protocol.get("validation_evaluation_rule_hash") != VALIDATION_EVALUATION_RULE_HASH:
    superseded_output_sha256 = {
        "metrics": previous_protocol.get("output_sha256", {}).get("metrics"),
        "guardrails": previous_protocol.get("output_sha256", {}).get("guardrails"),
    }
print("Train batches:", expcfg.SIMCA_CONCAT_REFIT_TRAIN_BATCHES)
print("Validation batches:", expcfg.SIMCA_CONCAT_REFIT_PROJECTION_BATCHES)
print("Forbidden batches:", expcfg.SIMCA_CONCAT_REFIT_FORBIDDEN_BATCHES)
print("Validation plan hash:", VALIDATION_PLAN_HASH)
print("Validation evaluation rule hash:", VALIDATION_EVALUATION_RULE_HASH)
print("Output:", OUTPUT_DIR)

Train batches: (1, 2)
Validation batches: (3,)
Forbidden batches: (4,)
Validation plan hash: 9e7b51fce2fe2c24b5ecc88387e1f54ec0304bea71b5ff3b1ed3e2c7cb2056e0
Validation evaluation rule hash: 424a9ec028c5f5d57f9a0035d1edceab300d617e91779d2a0afa3b5239f6750e
Output: C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04C_simca_concat_refit_8tracks_v3_non_noisy_all


## C — Vérification des verrous avant ouverture du batch 3

Le plan d'ablation 04B, le domaine 03B, l'éligibilité 03C et le verrou spatial sont validés avant le chargement du HDF5. Aucun paramètre manquant n'est complété par une valeur par défaut.

In [3]:
required_inputs = (
    CALIBRATED_PATH, CALIBRATION_DOMAIN_PATH, TRACK_CONTRACTS_PATH,
    PROJECTION_ELIGIBILITY_PATH, SPATIAL_METRICS_PATH,
    FRAGMENT_CLASSES_PATH, SPATIAL_LOCK_PATH, GRID_PARETO_PATH,
    GRID_PROTOCOL_PATH, OPTUNA_TRIALS_PATH, OPTUNA_CANDIDATES_PATH,
    OPTUNA_ABLATION_PATH, OPTUNA_PROTOCOL_PATH,
)
missing_inputs = [str(path) for path in required_inputs if not path.exists()]
if missing_inputs:
    raise FileNotFoundError(f"Missing upstream inputs: {missing_inputs}")

calibrated_df = load_parquet(CALIBRATED_PATH)
calibration_domain_df = load_parquet(CALIBRATION_DOMAIN_PATH)
track_contracts_df = load_parquet(TRACK_CONTRACTS_PATH)
projection_eligibility_df = load_parquet(PROJECTION_ELIGIBILITY_PATH)
spatial_calibration_metrics_df = load_parquet(SPATIAL_METRICS_PATH)
fragment_size_classes_df = load_parquet(FRAGMENT_CLASSES_PATH)
spatial_lock = json.loads(SPATIAL_LOCK_PATH.read_text(encoding="utf-8"))
grid_pareto_df = load_parquet(GRID_PARETO_PATH)
grid_protocol = json.loads(GRID_PROTOCOL_PATH.read_text(encoding="utf-8"))
optuna_trials_df = load_parquet(OPTUNA_TRIALS_PATH)
optuna_candidates_df = load_parquet(OPTUNA_CANDIDATES_PATH)
optuna_ablation_df = load_parquet(OPTUNA_ABLATION_PATH)
optuna_protocol = json.loads(OPTUNA_PROTOCOL_PATH.read_text(encoding="utf-8"))

grid_pareto_hash = sha256_file(GRID_PARETO_PATH)
if sha256_file(CALIBRATION_DOMAIN_PATH) != str(grid_protocol["input_sha256"]["03B_calibration_domain"]):
    raise RuntimeError("03B calibration-domain hash mismatch.")
if sha256_file(TRACK_CONTRACTS_PATH) != str(grid_protocol["input_sha256"]["03B_track_contracts"]):
    raise RuntimeError("03B track-contract hash mismatch.")
eligibility_hash = sha256_file(PROJECTION_ELIGIBILITY_PATH)
if eligibility_hash != str(grid_protocol["input_sha256"]["03C_projection_eligibility"]):
    raise RuntimeError("03C projection-eligibility hash mismatch.")
if eligibility_hash != str(optuna_protocol["input_sha256"]["03C_projection_eligibility"]):
    raise RuntimeError("03C and 04B eligibility provenance disagree.")
if grid_pareto_hash != str(grid_protocol["output_sha256"]["pareto_reference"]):
    raise RuntimeError("04A Pareto reference hash mismatch.")
if grid_pareto_hash != str(optuna_protocol["input_sha256"]["04A_pareto_reference"]):
    raise RuntimeError("04A and 04B Pareto provenance disagree.")
if bool(grid_protocol.get("weighted_score_used", True)):
    raise RuntimeError("04C requires the score-free 04A Pareto reference.")
if sha256_file(OPTUNA_TRIALS_PATH) != str(optuna_protocol["output_sha256"]["trials"]):
    raise RuntimeError("04B trials hash mismatch.")
if sha256_file(OPTUNA_CANDIDATES_PATH) != str(optuna_protocol["output_sha256"]["pareto_candidates"]):
    raise RuntimeError("04B Pareto candidates hash mismatch.")
expected_ablation_hash = str(optuna_protocol["ablation_plan"]["sha256"])
if sha256_file(OPTUNA_ABLATION_PATH) != expected_ablation_hash:
    raise RuntimeError("04B preregistered ablation plan hash mismatch.")
if not optuna_ablation_df["preregistered"].astype(bool).all():
    raise RuntimeError("04C cannot open batch 3 without a frozen ablation plan.")
if set(optuna_ablation_df["search_plan_hash"].astype(str)) != {str(optuna_protocol["search_plan_hash"])}:
    raise RuntimeError("04B ablation and search-plan hashes disagree.")
if bool(optuna_protocol.get("batch3_or_batch4_loaded", True)):
    raise RuntimeError("04B provenance does not prove a batches-1-2-only objective.")
spatial_lock_file_hash = sha256_file(SPATIAL_LOCK_PATH)
if spatial_lock_file_hash != str(grid_protocol["input_sha256"]["03C_spatial_postprocessing_lock"]):
    raise RuntimeError("03C spatial-lock file hash mismatch.")
if spatial_lock_file_hash != str(optuna_protocol["input_sha256"]["03C_spatial_lock"]):
    raise RuntimeError("03C and 04B spatial-lock provenance disagree.")
verify_spatial_postprocessing_lock(
    spatial_lock, spatial_calibration_metrics_df, fragment_size_classes_df
)
if set(track_contracts_df["evaluation_track"].astype(str)) != set(expcfg.SIMCA_EVALUATION_TRACKS):
    raise RuntimeError("The frozen 03B contract does not contain the eight tracks.")

candidate_pool_df = build_locked_validation_candidate_pool(
    calibrated_df, calibration_domain_df, grid_pareto_df,
    projection_eligibility_df, optuna_trials=optuna_trials_df,
    optuna_pareto_candidates=optuna_candidates_df,
)
CANDIDATE_POOL_HASH = sha256_dataframe(candidate_pool_df)
display(
    candidate_pool_df.groupby(
        ["track_id", "eligibility_status", "candidate_front"],
        dropna=False, sort=False,
    ).size().reset_index(name="n_candidates")
)
print("Frozen candidate pool:", len(candidate_pool_df))
print("Candidate pool hash:", CANDIDATE_POOL_HASH)

,track_id,eligibility_status,candidate_front,n_candidates
0,E1,eligible,protocol_pareto,6
1,E2,eligible_with_warning,protocol_pareto,327
2,E4,unsupported_domain_shift,diagnostic_pareto_unsupported_domain_shift,205
3,E5,eligible,protocol_pareto,3
4,E6,eligible,protocol_pareto,10
5,E7,eligible_with_warning,protocol_pareto,12
6,E8,unsupported_domain_shift,diagnostic_pareto_unsupported_domain_shift,381


Frozen candidate pool: 944
Candidate pool hash: 5c4e27edc185f7728d5e5fc9dfd89dad70d0a5be4923a6177611b312f984e178


## D — Chargement strict des batches 1–3

In [4]:
if not expcfg.SIMCA_CONCAT_REFIT_RUN:
    raise RuntimeError("SIMCA_CONCAT_REFIT_RUN=False: enable it to produce task-31 outputs.")
allowed_batches = tuple(sorted(set(
    expcfg.SIMCA_CONCAT_REFIT_TRAIN_BATCHES
    + expcfg.SIMCA_CONCAT_REFIT_PROJECTION_BATCHES
)))
if set(allowed_batches).intersection(expcfg.SIMCA_CONCAT_REFIT_FORBIDDEN_BATCHES):
    raise RuntimeError("A forbidden batch entered the 04C load contract.")
object_db, image_db = load_nir_uco_h5(
    DB_H5_PATH,
    reconstruct_heavy_object_arrays=expcfg.SIMCA_CONCAT_REFIT_RECONSTRUCT_HEAVY_OBJECT_ARRAYS,
    batches=allowed_batches,
)
loaded_batches = {
    int(record["batch"]) for record in object_db.values()
    if record.get("batch") is not None
}
if not loaded_batches.issubset(set(allowed_batches)):
    raise RuntimeError(f"Unexpected HDF5 batches: {sorted(loaded_batches)}")
if loaded_batches.intersection(expcfg.SIMCA_CONCAT_REFIT_FORBIDDEN_BATCHES):
    raise RuntimeError("Batch 4 was loaded before final selection.")
if USE_WAVELENGTH_WINDOW:
    object_db, image_db, wavelengths, _ = select_wavelength_range_from_database(
        object_db=object_db, image_db=image_db,
        min_nm=expcfg.WAVELENGTH_WINDOW_MIN_NM,
        max_nm=expcfg.WAVELENGTH_WINDOW_MAX_NM,
    )
else:
    wavelengths = np.asarray(next(iter(object_db.values()))["wavelengths"], dtype=float)
print("Loaded batches:", sorted(loaded_batches))

Loaded batches: [1, 2, 3]


## E — Tâche 31 : refit batches 1–2 et projection batch 3

Les prédictions continues sont matérialisées une seule fois par `projection_config_id`. Les décisions propres à chaque `calibration_id` sont ensuite appliquées avec les seuils 03B verrouillés.

In [5]:
checkpoint_context = {
    "validation_plan_hash": VALIDATION_PLAN_HASH,
    "candidate_pool_hash": CANDIDATE_POOL_HASH,
    "ablation_plan_hash": expected_ablation_hash,
    "spatial_lock_hash": str(spatial_lock["lock_sha256"]),
}
refit_kwargs = {
    "wavelengths": wavelengths,
    "train_batches": expcfg.SIMCA_CONCAT_REFIT_TRAIN_BATCHES,
    "projection_batches": expcfg.SIMCA_CONCAT_REFIT_PROJECTION_BATCHES,
    "target_class": expcfg.TARGET_CLASS,
    "non_target_label": expcfg.NON_TARGET_LABEL,
    "border_width": expcfg.SIMCA_CONCAT_REFIT_BORDER_WIDTH,
    "verbose": expcfg.SIMCA_CONCAT_REFIT_VERBOSE,
}
if expcfg.SIMCA_CONCAT_REFIT_CHECKPOINT_ENABLED:
    refit_outputs = run_locked_simca_validation_refit_checkpointed(
        candidate_pool_df, object_db=object_db, checkpoint_dir=CHECKPOINT_DIR,
        checkpoint_context=checkpoint_context,
        resume=expcfg.SIMCA_CONCAT_REFIT_RESUME_FROM_CHECKPOINT,
        **refit_kwargs,
    )
else:
    refit_outputs = run_locked_simca_validation_refit(
        candidate_pool_df, object_db=object_db, **refit_kwargs
    )
validation_object_predictions_df = refit_outputs["object_predictions"]
validation_pixel_predictions_df = refit_outputs["pixel_predictions"]
technical_errors_df = refit_outputs["technical_errors"]
validation_metrics_df = evaluate_locked_validation_predictions(
    candidate_pool_df, validation_object_predictions_df,
    validation_pixel_predictions_df, technical_errors=technical_errors_df,
)
overall_validation_metrics = validation_metrics_df.loc[
    validation_metrics_df["aggregation_level"].eq("overall")
    & validation_metrics_df["status"].eq("calculable")
].copy()
pixel_2way_metrics = overall_validation_metrics.loc[
    overall_validation_metrics["projection_level"].eq("pixel_projection")
    & overall_validation_metrics["decision_mode"].eq("2way")
]
if pd.to_numeric(pixel_2way_metrics["macro_image_balanced_accuracy"], errors="coerce").isna().any():
    raise RuntimeError("A calculable pixel 2-way candidate has no finite macro-image balanced accuracy.")
pixel_3way_metrics = overall_validation_metrics.loc[
    overall_validation_metrics["projection_level"].eq("pixel_projection")
    & overall_validation_metrics["decision_mode"].eq("3way")
]
if pd.to_numeric(pixel_3way_metrics["macro_image_decided_balanced_accuracy"], errors="coerce").isna().any():
    raise RuntimeError("A calculable pixel 3-way candidate has no finite macro-image decided balanced accuracy.")
display(
    validation_metrics_df.query("aggregation_level == 'overall'")
    .groupby(["track_id", "status"], dropna=False).size()
    .reset_index(name="n_candidates")
)

,track_id,status,n_candidates
0,E1,calculable,6
1,E2,calculable,327
2,E3,not_evaluable_no_calibrated_candidate,1
3,E4,calculable,205
4,E5,calculable,3
5,E6,calculable,10
6,E7,calculable,12
7,E8,calculable,381


## F — Tâche 32 : cartes pixels et fragments

La marge continue reste dans `validation_pixel_predictions.parquet`. Le manifeste encode les couches valide/arrière-plan, cible brute, incertitude immuable, cible post-traitée et vérité. Sur les images pures du batch 3, la classe pixel est exacte dans le ROI segmenté ; les labels de segmentation servent de vérité composante afin de ne pas fusionner des objets adjacents.

In [6]:
spatial_outputs = build_locked_spatial_validation_outputs(
    candidate_pool_df, validation_pixel_predictions_df, image_db, spatial_lock
)
pixel_maps_manifest_df = spatial_outputs["pixel_maps_manifest"]
spatial_components_df = spatial_outputs["spatial_components"]
spatial_component_metrics_df = spatial_outputs["spatial_component_metrics"]
if len(pixel_maps_manifest_df):
    if not pixel_maps_manifest_df["truth_level"].eq(expcfg.SIMCA_CONCAT_REFIT_TRUTH_SOURCE).all():
        raise RuntimeError("Batch-3 pure-image truth was not recorded as exact.")
    if not set(spatial_component_metrics_df["map_variant"].astype(str)) >= {"raw", "locked_postprocessed"}:
        raise RuntimeError("Spatial evaluation requires raw and postprocessed maps.")
display(
    spatial_component_metrics_df.query("aggregation_level == 'overall'")
    .groupby(["track_id", "map_variant"], dropna=False).size()
    .reset_index(name="n_candidates")
)

,track_id,map_variant,n_candidates
0,E4,locked_postprocessed,205
1,E4,raw,205
2,E7,locked_postprocessed,12
3,E7,raw,12
4,E8,locked_postprocessed,381
5,E8,raw,381


## G — Tâche 33 : garde-fous de validation explicables

In [7]:
validation_guardrails_df = build_validation_guardrails(
    candidate_pool_df, validation_metrics_df,
    spatial_component_metrics=spatial_component_metrics_df,
)
if any("score" in column.lower() for column in validation_guardrails_df.columns):
    raise RuntimeError("A composite score entered the validation guardrails.")
threshold_checks = validation_guardrails_df.loc[
    validation_guardrails_df["check_status"].isin(["pass", "fail"])
]
non_finite_threshold_checks = threshold_checks.loc[
    ~np.isfinite(pd.to_numeric(threshold_checks["observed_value"], errors="coerce"))
]
if len(non_finite_threshold_checks):
    raise RuntimeError("A scientific pass/fail guardrail contains a non-finite metric.")
status_table = (
    validation_guardrails_df[["track_id", "validation_candidate_id", "candidate_status"]]
    .drop_duplicates()
    .groupby(["track_id", "candidate_status"], dropna=False)
    .size().reset_index(name="n_candidates")
)
display(status_table)
unsupported_tracks = set(
    projection_eligibility_df.loc[
        projection_eligibility_df["eligibility_status"].eq("unsupported_domain_shift"),
        "track_id",
    ].astype(str)
)
observed_unsupported = set(
    validation_guardrails_df.loc[
        validation_guardrails_df["candidate_status"].eq("unsupported_domain_shift_diagnostic"),
        "track_id",
    ].astype(str)
)
if not unsupported_tracks.issubset(observed_unsupported):
    raise RuntimeError("An unsupported-domain-shift track was removed silently.")

,track_id,candidate_status,n_candidates
0,E1,calculable_but_not_acceptable,2
1,E1,pass,4
2,E2,calculable_but_not_acceptable,30
3,E2,pass,297
4,E3,not_evaluable_no_calibrated_candidate,1
5,E4,unsupported_domain_shift_diagnostic,205
6,E5,pass,3
7,E6,calculable_but_not_acceptable,3
8,E6,pass,7
9,E7,calculable_but_not_acceptable,6


## H — Sauvegarde du contrat des tâches 31–33

In [8]:
tables_to_save = {
    "object_predictions": validation_object_predictions_df.reindex(columns=expcfg.SIMCA_VALIDATION_OBJECT_PREDICTION_COLUMNS),
    "pixel_predictions": validation_pixel_predictions_df.reindex(columns=expcfg.SIMCA_VALIDATION_PIXEL_PREDICTION_COLUMNS),
    "metrics": validation_metrics_df.reindex(columns=expcfg.SIMCA_VALIDATION_METRIC_COLUMNS),
    "pixel_maps_manifest": pixel_maps_manifest_df.reindex(columns=expcfg.SIMCA_PIXEL_MAP_MANIFEST_COLUMNS),
    "spatial_components": spatial_components_df.reindex(columns=expcfg.SIMCA_SPATIAL_COMPONENT_COLUMNS),
    "spatial_component_metrics": spatial_component_metrics_df.reindex(columns=expcfg.SIMCA_SPATIAL_COMPONENT_METRIC_COLUMNS),
    "guardrails": validation_guardrails_df.reindex(columns=expcfg.SIMCA_VALIDATION_GUARDRAIL_COLUMNS),
}
for name, table in tables_to_save.items():
    save_parquet(table, OUTPUT_PATHS[name])

protocol = {
    "notebook": "04C_simca_concat_refit",
    "tasks": [31, 32, 33],
    "results_tag": RESULTS_TAG,
    "protocol_version": expcfg.PROTOCOL_VERSION,
    "validation_plan_hash": VALIDATION_PLAN_HASH,
    "validation_evaluation_rule_version": expcfg.SIMCA_CONCAT_REFIT_EVALUATION_RULE_VERSION,
    "validation_evaluation_rule_hash": VALIDATION_EVALUATION_RULE_HASH,
    "candidate_pool_hash": CANDIDATE_POOL_HASH,
    "candidate_policy": expcfg.SIMCA_CONCAT_REFIT_CANDIDATE_POLICY,
    "scientific_score_or_rank_filter_applied": False,
    "optuna_role": "provenance_annotation_only",
    "train_batches": list(expcfg.SIMCA_CONCAT_REFIT_TRAIN_BATCHES),
    "projection_batches": list(expcfg.SIMCA_CONCAT_REFIT_PROJECTION_BATCHES),
    "forbidden_batches_not_loaded": list(expcfg.SIMCA_CONCAT_REFIT_FORBIDDEN_BATCHES),
    "threshold_policy": "fixed_from_03B_no_batch3_recalibration",
    "guardrail_metric_policy": {
        "overall_object": "pooled_class_balanced_metrics",
        "overall_pixel": "class_conditional_macro_image_metrics",
        "worst_image": "class_conditional_risks_with_scope_specific_limits",
        "coverage_rate": "reported_exact_complement_not_second_blocking_guardrail",
        "single_class_image_balanced_accuracy": "not_applicable",
    },
    "guardrail_amendment": expcfg.SIMCA_CONCAT_REFIT_EVALUATION_AMENDMENT,
    "supersedes_output_sha256": superseded_output_sha256,
    "random_state_policy": "exact_per_calibration_id_from_03B",
    "spatial_lock_sha256": str(spatial_lock["lock_sha256"]),
    "spatial_truth": expcfg.SIMCA_CONCAT_REFIT_TRUTH_SOURCE,
    "uncertainty_policy": "preserve_as_distinct_immutable_layer",
    "smallest_fragment_guardrail": (
        "diagnostic_only_no_prespecified_threshold"
        if expcfg.SIMCA_CONCAT_REFIT_SMALLEST_FRAGMENT_RECALL_MIN is None
        else float(expcfg.SIMCA_CONCAT_REFIT_SMALLEST_FRAGMENT_RECALL_MIN)
    ),
    "ablation_plan_verified_before_batch3_load": True,
    "ablation_plan_sha256": expected_ablation_hash,
    "optuna_search_plan_hash": str(optuna_protocol["search_plan_hash"]),
    "n_frozen_candidates": int(len(candidate_pool_df)),
    "n_unique_fits": int(candidate_pool_df["fit_config_id"].nunique()),
    "n_unique_projections": int(candidate_pool_df["projection_config_id"].nunique()),
    "n_technical_errors": int(len(technical_errors_df)),
    "input_sha256": {
        str(path.relative_to(PROJECT_ROOT)): sha256_file(path)
        for path in required_inputs
    },
    "output_sha256": {
        name: sha256_file(OUTPUT_PATHS[name]) for name in tables_to_save
    },
}
OUTPUT_PATHS["protocol"].write_text(
    json.dumps(protocol, indent=2, ensure_ascii=False), encoding="utf-8"
)
print("Saved:")
for path in OUTPUT_PATHS.values():
    print(" -", path)

Saved:
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04C_simca_concat_refit_8tracks_v3_non_noisy_all\validation_object_predictions.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04C_simca_concat_refit_8tracks_v3_non_noisy_all\validation_pixel_predictions.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04C_simca_concat_refit_8tracks_v3_non_noisy_all\validation_metrics.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04C_simca_concat_refit_8tracks_v3_non_noisy_all\pixel_maps_manifest.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04C_simca_concat_refit_8tracks_v3_non_noisy_all\spatial_components.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04C_simca_concat_refit_8tracks_v3_non_noisy_all\spatial_component_metrics.parquet
 - C:\Users\alixg\OneDrive - Université Paris-Dauphine\hsi_nuts\results\04C_simca_

## Contrat de sortie

- `validation_object_predictions.parquet` et `validation_pixel_predictions.parquet` conservent les diagnostics continus sans duplication par seuil.
- `validation_metrics.parquet` contient les résultats globaux et par image, les intervalles, les signatures et les groupes d'équivalence sans suppression.
- `pixel_maps_manifest.parquet` conserve les couches brutes, incertaines, post-traitées et de vérité dans un encodage compact et versionné.
- `spatial_components.parquet` et `spatial_component_metrics.parquet` documentent tailles, associations IoU, split/merge et rappel des petites classes d'aire.
- `validation_guardrails.parquet` attribue un statut explicable à chaque candidat et conserve les tracks non soutenus comme diagnostics scientifiques.
- Sur les images pures monoclasse, les risques `worst_image` utilisent les limites par image du profil de risque actif ; la balanced accuracy est calculée entre classes au niveau global ou macro-image, jamais dans une image monoclasse. La couverture reste reportée comme complément exact de l'incertitude, sans constituer un second garde-fou bloquant.